# Encounter type — one head, one page at a time (Colab first)

Inpatient · Outpatient (F2F) · Outpatient (Evisit) · Outpatient (Home Visit) · Not Available, one page at a time. **Colab:** Runtime → Change runtime type → **GPU**, then edit `BASE_PATH` only.

Put these in one Drive folder (`BASE_PATH`):

* `train_encounter.py`, `predict_encounter.py`, `requirements.txt` — from `training/bert-training-encounter/`
* `encounter_tags.jsonl` — from `Training/encounter/` (2,750 pages)
* `mock_pages.jsonl` — optional, from `Training/mock/` (synthetic; with `USE_MOCK = True` they train, never score)

What is not learned: `train: false` rows (the client default "outpatient, no modality stated → Evisit"; it stays a rule after the model). Labels inherited from another page of the visit train at half weight. The model is written to `BASE_PATH/encounter/` after every improving epoch. Page text is never printed.

**Target (2026-10-25):** `--target page` (the default) learns `encounter_on_page`: the setting only when the page itself shows it, else **No evidence on page**, a sixth class. The visit is applied to the whole document afterwards, so a page that says nothing abstains instead of guessing. `--target visit` learns `encounter_type` (the first run's five classes). Also put `encounter_test_pages.jsonl` and `check_encounter_tests.py` in the Drive folder for the last check.

**Balancing (2026-10-25):** classes under 100 training pages are oversampled (Evisit ~x2, Home Visit ~x2.8) and `synthetic_pages.jsonl` (80 invented Evisit / Home Visit pages, made by `make_synthetic_pages.py`) is added to the training split with the mock pages. Upload it with the other files.

In [ ]:
import json, os, sys, subprocess
from pathlib import Path

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/encounter-training')  # <- the only thing to edit
    CODE, DATA, OUTPUT = BASE_PATH, BASE_PATH / 'encounter_tags.jsonl', BASE_PATH / 'encounter'
    MOCK = BASE_PATH / 'mock_pages.jsonl'
else:
    CODE = Path.cwd()                                              # run from training/bert-training-encounter
    DATA, OUTPUT = CODE / 'data' / 'encounter_tags.jsonl', CODE / 'output' / 'encounter'
    MOCK = CODE / 'data' / 'mock_pages.jsonl'
USE_MOCK = True        # the mock pages (Training/mock), training split only
USE_SYNTHETIC = True   # 80 invented Evisit / Home Visit pages (make_synthetic_pages.py), training split only
SYNTHETIC = CODE / 'synthetic_pages.jsonl'
EPOCHS = 20

# A Hugging Face name, or a folder holding the weights where Hugging Face is blocked.
BASE_MODEL = 'distilbert-base-uncased'

for name in ('train_encounter.py', 'predict_encounter.py'):
    assert (CODE / name).is_file(), f'{name} is missing from {CODE}'
assert DATA.is_file(), f'no training data at {DATA}'
assert not USE_MOCK or MOCK.is_file(), f'USE_MOCK is on but there is no {MOCK}'
assert not USE_SYNTHETIC or SYNTHETIC.is_file(), f'USE_SYNTHETIC is on but there is no {SYNTHETIC}'
sys.path.insert(0, str(CODE))
print('code:', CODE, '| data:', DATA, '| output:', OUTPUT)

In [ ]:
if ON_COLAB:
    %pip install -q -r {str(CODE / 'requirements.txt')}

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
def train(*args):
    """Run train_encounter.py with these paths; its output streams below."""
    cmd = [sys.executable, str(CODE / 'train_encounter.py'), '--data', str(DATA), '--output', str(OUTPUT),
           '--base-model', BASE_MODEL, *(['--with-mock', *([str(MOCK)] if USE_MOCK else []), *([str(SYNTHETIC)] if USE_SYNTHETIC else [])]
             if USE_MOCK or USE_SYNTHETIC else []), *args]
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

Compatibility check: class and basis counts, the chart split, two training steps and a minutes-per-epoch estimate. Saves nothing.

In [ ]:
train('--check')

A run: 20 epochs (`EPOCHS`). The output folder keeps the best epoch by validation macro F1 (so Home Visit and Evisit count as much as F2F). At the end: per-class recall and precision on the held-out test charts, page by page and visit by visit (the pipeline's vote: first page twice, Not Available only when every page says so), page accuracy by label basis, and what the model says on the client-default rows. Options: `--inherited-weight` (0.5; 0 leaves inherited labels out), `--test-fraction`, `--lr`.

In [ ]:
train('--epochs', str(EPOCHS))

In [ ]:
import pandas as pd
pd.read_csv(OUTPUT / 'training_log.csv')

In [ ]:
print(json.dumps(json.loads((OUTPUT / 'metrics_summary.json').read_text()), indent=2))

In [ ]:
display(pd.read_csv(OUTPUT / 'class_metrics_pages.csv'))
pd.read_csv(OUTPUT / 'confusion_pages.csv', index_col=0)

In [ ]:
display(pd.read_csv(OUTPUT / 'class_metrics_visits.csv'))
pd.read_csv(OUTPUT / 'confusion_visits.csv', index_col=0)

Errors (no text), most confident first: often label errors to fix in the annotation tool.

In [ ]:
errors = pd.read_csv(OUTPUT / 'errors.csv')
print(errors.groupby(['label', 'pred']).size().sort_values(ascending=False).head(15))
errors.sort_values('confidence', ascending=False).head(30)

## Encounter test pages

22 made-up pages with the expected visit setting (several are traps for a keyword engine). A page-level model may answer *No evidence on page* where the expected answer is the visit's: that is a fail here, and in the pipeline the rules and the family default decide such a page. ENC_22 (the client default) is reported, not scored.

In [ ]:
import csv
from predict_encounter import EncounterClassifier
tests = [json.loads(l) for l in (CODE / 'encounter_test_pages.jsonl').read_text().splitlines() if l.strip()]
clf = EncounterClassifier(OUTPUT)
answers = clf.predict_many([t['raw_text'] for t in tests])
with open(OUTPUT / 'encounter_test_predictions.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['id', 'encounter_type'])
    w.writerows([t['id'], a['encounter_type']] for t, a in zip(tests, answers))
result = subprocess.run([sys.executable, str(CODE / 'check_encounter_tests.py'), '--pages', str(CODE / 'encounter_test_pages.jsonl'),
                         '--pred', str(OUTPUT / 'encounter_test_predictions.csv')], capture_output=True, text=True)
print(result.stdout, result.stderr)

## Inference check

Paste a page's text.

In [ ]:
from predict_encounter import EncounterClassifier
clf = EncounterClassifier(OUTPUT)
TEXT = """
paste page text here
"""
clf.show(TEXT)